In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd 

import itertools
from collections import defaultdict

from scripts.ss_graphgen import *

In [2]:
dttz = pd.read_csv("../data/student_portuguese.csv")
dtz = dttz.drop_duplicates().reset_index(drop=True)

print("Original number of rows: ", len(dtz))
print(" ")

seln = 500

dtx = dtz.sample(n=seln, replace=False, random_state=42).reset_index(drop=True)
dt = dtx.drop_duplicates().reset_index(drop=True)
dt


Original number of rows:  649
 


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,pass
0,1,1,18,1,0,1,4,4,4,4,...,1,0,3,2,4,1,4,2,4,1
1,0,0,16,1,0,0,3,1,3,2,...,1,0,2,3,3,2,2,4,2,0
2,1,0,18,1,0,1,4,4,4,4,...,1,0,4,3,5,1,2,1,0,1
3,1,1,16,0,1,0,4,4,0,2,...,0,0,5,3,2,1,3,2,5,0
4,0,0,15,0,0,1,1,1,2,2,...,1,1,3,3,4,2,4,5,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,1,0,18,0,0,1,1,1,0,2,...,1,1,4,3,2,1,1,5,9,0
496,0,1,16,1,1,1,2,1,0,2,...,0,1,4,4,4,3,5,5,6,0
497,0,0,17,1,1,1,2,2,3,2,...,1,1,4,4,4,2,3,5,6,1
498,0,1,16,1,0,1,2,1,0,2,...,0,0,3,2,1,1,1,2,4,0


In [3]:
target_column = "pass"

X = dt.drop([target_column], axis=1).values
y = dt[target_column].values
y = np.where(y == 1, 1, -1) 

s = dt["sex"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)



In [4]:
##########
np.random.seed(42)
n = X.shape[0]
indices = np.random.permutation(n)
split = int(0.9 * n)
lhs_idx = indices[:split]
rhs_idx = indices[split:]

lhs_idx_negative = lhs_idx[y[lhs_idx] == -1]
X_LHS_random = X[lhs_idx_negative]
X_RHS_random = X[rhs_idx]
y_RHS_random = y[rhs_idx]

s_LHS_random = s[lhs_idx_negative]
s_RHS_random = s[rhs_idx]

xxneg = y[lhs_idx]

len(lhs_idx_negative), len(lhs_idx), len(rhs_idx), np.sum(xxneg == -1), np.sum(xxneg == 1)


(224, 450, 50, 224, 226)

In [5]:
##########
##########
edges_random, labels_random, graph_random = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                                XRHS=X_RHS_random, 
                                                                yRHS=y_RHS_random, 
                                                                sLHS=s_LHS_random, 
                                                                sRHS=s_RHS_random, 
                                                                k_min=1, 
                                                                k_max=10)

r_edges_random = graph_random['edges']
r_labels_random = graph_random['labels']

r_graph_structure_random = {
    f"x_{x}": [f"t_{t} ({'+' if r_labels_random[t] == 1 else '-'})" for t in neighbors]
    for x, neighbors in r_edges_random.items()
}

print("Random bipartite graph")
for agent, targets in list(r_graph_structure_random.items())[:10]: 
    print(f"{agent}: {targets}")
    
    
print (" ")
print("positive ", {t for t, l in r_labels_random.items() if l == 1})
print("negative ", {t for t, l in r_labels_random.items() if l == -1})
    
graph_random['n'], graph_random['m']


Random bipartite graph
x_0: ['t_29 (+)', 't_40 (-)', 't_45 (-)', 't_8 (-)', 't_48 (-)', 't_31 (-)', 't_46 (-)', 't_33 (-)', 't_42 (+)', 't_6 (+)']
x_1: ['t_37 (-)', 't_5 (-)', 't_6 (+)', 't_26 (+)', 't_45 (-)', 't_42 (+)', 't_17 (-)', 't_28 (+)', 't_27 (+)', 't_18 (+)']
x_2: ['t_47 (+)', 't_42 (+)', 't_46 (-)', 't_34 (-)', 't_0 (+)', 't_28 (+)', 't_41 (+)', 't_6 (+)', 't_30 (+)', 't_14 (+)']
x_3: ['t_31 (-)', 't_48 (-)', 't_7 (-)', 't_41 (+)', 't_14 (+)', 't_32 (-)', 't_23 (+)', 't_40 (-)', 't_24 (+)', 't_34 (-)']
x_4: ['t_8 (-)', 't_4 (-)', 't_37 (-)', 't_46 (-)', 't_42 (+)', 't_47 (+)', 't_5 (-)', 't_34 (-)', 't_41 (+)', 't_44 (+)']
x_5: ['t_28 (+)', 't_26 (+)', 't_41 (+)', 't_27 (+)', 't_42 (+)', 't_37 (-)', 't_47 (+)', 't_32 (-)', 't_12 (+)', 't_7 (-)']
x_6: ['t_9 (-)', 't_24 (+)', 't_47 (+)', 't_33 (-)', 't_27 (+)', 't_29 (+)', 't_32 (-)', 't_44 (+)', 't_42 (+)', 't_21 (-)']
x_7: ['t_45 (-)', 't_46 (-)', 't_24 (+)', 't_34 (-)', 't_47 (+)', 't_7 (-)', 't_23 (+)', 't_5 (-)', 't_28 (

(224, 50)

In [6]:
##########
edges_random, labels_random, graph_random = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                                XRHS=X_RHS_random, 
                                                                yRHS=y_RHS_random,
                                                                sLHS=s_LHS_random, 
                                                                sRHS=s_RHS_random, 
                                                                threshold=0.2, 
                                                                topk=False, 
                                                                thresh=True)


r_edges_random = graph_random['edges']
r_labels_random = graph_random['labels']

r_graph_structure_random = {
    f"x_{x}": [f"t_{t} ({'+' if r_labels_random[t] == 1 else '-'})" for t in neighbors]
    for x, neighbors in r_edges_random.items()
}

print("Random bipartite graph")
for agent, targets in list(r_graph_structure_random.items())[:10]: 
    print(f"{agent}: {targets}")
    
    
print (" ")
print("positive ", {t for t, l in r_labels_random.items() if l == 1})
print("negative ", {t for t, l in r_labels_random.items() if l == -1})
    
graph_random['n'], graph_random['m']


Random bipartite graph
x_0: []
x_1: []
x_2: []
x_3: []
x_4: []
x_5: []
x_6: []
x_7: []
x_8: []
x_9: []
 
positive  set()
negative  set()


(224, 0)

## General

In [7]:
#############
##knn##
#############
graphsRandom1 = []
graphsinfor = []
graphid = 0
for kmax1 in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    _, _, graphx_random1 = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                XRHS=X_RHS_random, 
                                                yRHS=y_RHS_random, 
                                                sLHS=s_LHS_random, 
                                                sRHS=s_RHS_random, 
                                                k_min=1, 
                                                k_max=kmax1)
    
    graphx_random1.update({"k_max": kmax1, "knn": "yes", "thresh": "no"})
    graphsRandom1.append(graphx_random1)
    
    
    ########
    avg_left_deg, avg_left_pos_deg, avg_left_neg_deg, avg_right_deg, avg_overlap, \
    avg_pos_overlap, avg_neg_overlap, only_pos, only_neg, empty_adj, unipos  = getconnectivity_info(graphx_random1["edges"], graphx_random1["labels"])

    graphsinfor.append({
        "Dataset (d)": "Student-portguese ("+ str(X.shape[1]) + ")",
        "kmax": kmax1, 
        "n": graphx_random1['n'],
        "m": graphx_random1['m'],
        "#+ves": len({t for t, l in graphx_random1['labels'].items() if l == 1}),
        "#-ves": len({t for t, l in graphx_random1['labels'].items() if l == -1}), 
        "avg LHSd": avg_left_deg, 
        "avg LHS+d": avg_left_pos_deg, 
        "avg LHS-d": avg_left_neg_deg,  
        "avg RHSd": avg_right_deg,         
        "avg overlap": avg_overlap,
        "avg overlap+": avg_pos_overlap,
        "avg overlap-": avg_neg_overlap,
        "only+Ns": only_pos,
        "only-Ns": only_neg,
        "emptyNs": empty_adj,
        "uni+": unipos,       
        "graphID": graphid
    })
    
    graphid += 1

######
randomgraphsinfo = pd.DataFrame(graphsinfor)   

######
np.save("../graphs/studentportuguese_knn_random.npy", np.array(graphsRandom1, dtype=object), allow_pickle=True)
randomgraphsinfo.to_csv("../graphs/studentportuguese_knn_graphsummary.npy", index=False)
######
randomgraphsinfo
 


,Dataset (d),kmax,n,m,#+ves,#-ves,avg LHSd,avg LHS+d,avg LHS-d,avg RHSd,avg overlap,avg overlap+,avg overlap-,only+Ns,only-Ns,emptyNs,uni+,graphID
0,Student-portguese (30),1,224,46,20,26,1.0,0.486607,0.513393,4.869565,0.033352,0.017417,0.015935,109,115,0,0,0
1,Student-portguese (30),2,224,48,20,28,2.0,0.982143,1.017857,9.333333,0.140415,0.078075,0.062340,70,74,0,0,1
2,Student-portguese (30),3,224,48,20,28,3.0,1.540179,1.459821,14.000000,0.301650,0.179012,0.122638,38,36,0,0,2
3,Student-portguese (30),4,224,49,21,28,4.0,2.120536,1.879464,18.285714,0.528828,0.324952,0.203876,26,22,0,0,3
4,Student-portguese (30),5,224,50,22,28,5.0,2.656250,2.343750,22.400000,0.819827,0.505445,0.314382,17,13,0,0,4
5,Student-portguese (30),6,224,50,22,28,6.0,3.245536,2.754464,26.880000,1.146621,0.726297,0.420324,12,8,0,0,5
6,Student-portguese (30),7,224,50,22,28,7.0,3.781250,3.218750,31.360000,1.535714,0.975336,0.560378,3,6,0,0,6
7,Student-portguese (30),8,224,50,22,28,8.0,4.348214,3.651786,35.840000,1.973374,1.259009,0.714366,3,2,0,0,7
8,Student-portguese (30),9,224,50,22,28,9.0,4.955357,4.044643,40.320000,2.506246,1.621076,0.885170,1,1,0,0,8
9,Student-portguese (30),10,224,50,22,28,10.0,5.571429,4.428571,44.800000,3.055133,1.991832,1.063301,1,0,0,0,9


In [8]:
#############
##thresh##
#############
graphsRandom11 = []
graphsinfor11 = []
graphid11 = 0
for t11 in [4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10]:
    _, _, graphx_random11 = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                XRHS=X_RHS_random, 
                                                yRHS=y_RHS_random, 
                                                sLHS=s_LHS_random, 
                                                sRHS=s_RHS_random, 
                                                threshold=t11, 
                                                topk=False, 
                                                thresh=True)
    
    graphx_random11.update({"threshold": t11, "knn": "no", "thresh": "yes"})
    graphsRandom11.append(graphx_random11)

    ########
    avg_left_deg11, avg_left_pos_deg11, avg_left_neg_deg11, avg_right_deg11, avg_overlap11, \
    avg_pos_overlap11, avg_neg_overlap11, only_pos11, only_neg11, empty_adj11, unipos11  = getconnectivity_info(graphx_random11["edges"], graphx_random11["labels"])


    graphsinfor11.append({
        "Dataset (d)": "Student-portguese ("+ str(X.shape[1]) + ")",
        "r": t11, 
        "n": graphx_random11['n'],
        "m": graphx_random11['m'],
        "#+ves": len({t for t, l in graphx_random11['labels'].items() if l == 1}),
        "#-ves": len({t for t, l in graphx_random11['labels'].items() if l == -1}), 
        "avg LHSd": avg_left_deg11, 
        "avg LHS+d": avg_left_pos_deg11, 
        "avg LHS-d": avg_left_neg_deg11,  
        "avg RHSd": avg_right_deg11,         
        "avg overlap": avg_overlap11,
        "avg overlap+": avg_pos_overlap11,
        "avg overlap-": avg_neg_overlap11,  
        "only+Ns": only_pos11,
        "only-Ns": only_neg11,
        "emptyNs": empty_adj11,
        "uni+": unipos11,
        "graphID": graphid11
    })
    
    graphid11 += 1

#######
randomgraphsinfo11 = pd.DataFrame(graphsinfor11)   
     
    
#######   
np.save("../graphs/studentportuguese_thresh_random.npy", np.array(graphsRandom11, dtype=object), allow_pickle=True)
randomgraphsinfo11.to_csv("../graphs/studentportuguese_thresh_graphsummary.npy", index=False)
#######
randomgraphsinfo11


,Dataset (d),r,n,m,#+ves,#-ves,avg LHSd,avg LHS+d,avg LHS-d,avg RHSd,avg overlap,avg overlap+,avg overlap-,only+Ns,only-Ns,emptyNs,uni+,graphID
0,Student-portguese (30),4.0,224,6,5,1,0.035714,0.031250,0.004464,1.333333,0.001810,0.001810,0.000000,4,0,219,2,0
1,Student-portguese (30),4.5,224,22,13,9,0.214286,0.165179,0.049107,2.181818,0.011933,0.011592,0.000341,18,6,196,0,1
2,Student-portguese (30),5.0,224,32,18,14,0.665179,0.540179,0.125000,4.656250,0.059369,0.057328,0.002041,32,8,169,0,2
3,Student-portguese (30),5.5,224,39,20,19,1.500000,1.044643,0.455357,8.615385,0.146890,0.122548,0.024342,30,28,129,0,3
4,Student-portguese (30),6.0,224,44,20,24,3.433036,2.129464,1.303571,17.477273,0.470476,0.328889,0.141587,26,40,77,0,4
5,Student-portguese (30),6.5,224,48,21,27,7.116071,4.312500,2.803571,33.208333,1.655986,1.134377,0.521610,15,29,41,0,5
6,Student-portguese (30),7.0,224,49,21,28,12.691964,7.169643,5.522321,58.020408,4.525781,2.862906,1.662874,6,22,20,0,6
7,Student-portguese (30),7.5,224,50,22,28,19.758929,10.526786,9.232143,88.520000,9.798838,5.790984,4.007854,4,16,7,0,7
8,Student-portguese (30),8.0,224,50,22,28,27.272321,13.718750,13.553571,122.180000,17.218596,9.356865,7.861731,1,7,3,0,8
9,Student-portguese (30),8.5,224,50,22,28,34.062500,16.522321,17.540179,152.600000,25.435338,13.162676,12.272662,0,3,1,0,9
